# 第一步：EDA —— 摸清楚数据长什么样

项目：比特币交易欺诈检测（Elliptic 数据集）

这个 notebook 只做一件事：把数据下载下来，看清楚它长什么样，不训练任何模型。

## 1. 安装依赖

In [ ]:
!pip install torch_geometric -q

## 2. 挂载 Google Drive

把数据存进 Drive，下次开新 session 不用重新下载。第一次运行会弹窗要求登录授权，同意就行。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 下载并加载数据集

In [ ]:
from torch_geometric.datasets import EllipticBitcoinDataset

dataset = EllipticBitcoinDataset(root='/content/drive/MyDrive/data-mining/elliptic')
data = dataset[0]
print(data)

跑完这格应该看到类似 `Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], test_mask=[203769])` 的输出：
20万+ 个节点（交易）、23万+ 条边（资金流向）、每个节点165个特征。

## 4. 看节点特征长什么样（表格形式）

In [ ]:
import pandas as pd

feature_df = pd.DataFrame(data.x.numpy())
print("节点数、特征数：", feature_df.shape)
feature_df.head()

165列里，前94列是“本地特征”（这笔交易自己的信息：输入输出数量、手续费等），
后72列是“聚合特征”（跟它直接相连的1跳邻居交易的统计值：最大/最小/标准差）。
官方没有公布每一列具体对应什么，这是正常的，不是你漏看了什么。

## 5. 看标签分布

0 = 正常（licit），1 = 欺诈（illicit），2 = 未标注（unknown）

In [ ]:
import torch

labels, counts = torch.unique(data.y, return_counts=True)
for l, c in zip(labels.tolist(), counts.tolist()):
    print(f"标签 {l}：{c} 个节点")

## 6. 看边长什么样（资金流向）

In [ ]:
print("边的数量：", data.edge_index.shape[1])
data.edge_index[:, :5]

结果是两行数字，每一列是一条边：上面一行是钱转出的交易（起点），下面一行是收到钱的交易（终点）。
比如某一列是 `[5, 9]`，意思是节点5的钱转给了节点9，节点5和节点9互为“周围”（1跳邻居）。

## 7. 类别不平衡可视化

In [ ]:
import matplotlib.pyplot as plt

names = ['正常(0)', '欺诈(1)', '未标注(2)']
plt.bar(names, counts.tolist())
plt.title('各类别节点数量')
plt.show()

## 8. 训练集 / 测试集规模

时间步（1~49）本身没有保留在 `data.x` 里，PyG 内部已经用它算好了两个开关：
`train_mask`（早期时间步，time_step < 35）和 `test_mask`（晚期时间步，time_step >= 35），
两个都已经自动排除了标签为2（未标注）的节点，不用你自己重新实现按时间切分的逻辑。

In [ ]:
print("训练集节点数（早期时间步，已排除未标注）：", data.train_mask.sum().item())
print("测试集节点数（晚期时间步，已排除未标注）：", data.test_mask.sum().item())

## 9. 检查缺失值

In [ ]:
print("特征里有没有缺失值：", torch.isnan(data.x).any().item())

## 小结

- 203,769 个节点（交易），234,355 条边（资金流向）
- 165维特征：94维本地特征 + 72维1跳邻居聚合特征，官方未公布逐列含义
- 标签严重不平衡：欺诈仅占已标注节点的一小部分，多数节点未标注
- 时间步 1~49，PyG 已内置按时间切分的 `train_mask` / `test_mask`，评估时不能随机切分，避免用未来数据训练
- 无缺失值

下一步：`02_baseline.ipynb`，用这里的 `train_mask` / `test_mask` 跑决策树和 SVM 基线。